# Chapter 3: Implementing Self-Attention

## 1. Calculate attention scores for one query

Each row is a vector representation of one token. We select the second row as the query:

```text
query = [0.55, 0.87, 0.66]
```

For every input vector, the attention score is its dot product with the query:

$$\text{score}(x_i, q) = x_i \cdot q = \sum_k x_{i,k}q_k$$

The table shows every multiplication and the addition that produces each score. A larger score means stronger alignment with the query.

| Token | Input vector | Multiplications | Addition | Score |
|---|---|---|---:|---:|
| Your | `[0.43, 0.15, 0.89]` | `0.43×0.55=0.2365`<br>`0.15×0.87=0.1305`<br>`0.89×0.66=0.5874` | `0.2365+0.1305+0.5874` | **0.9544** |
| journey | `[0.55, 0.87, 0.66]` | `0.55×0.55=0.3025`<br>`0.87×0.87=0.7569`<br>`0.66×0.66=0.4356` | `0.3025+0.7569+0.4356` | **1.4950** |
| starts | `[0.57, 0.85, 0.64]` | `0.57×0.55=0.3135`<br>`0.85×0.87=0.7395`<br>`0.64×0.66=0.4224` | `0.3135+0.7395+0.4224` | **1.4754** |
| with | `[0.22, 0.58, 0.33]` | `0.22×0.55=0.1210`<br>`0.58×0.87=0.5046`<br>`0.33×0.66=0.2178` | `0.1210+0.5046+0.2178` | **0.8434** |
| one | `[0.77, 0.25, 0.10]` | `0.77×0.55=0.4235`<br>`0.25×0.87=0.2175`<br>`0.10×0.66=0.0660` | `0.4235+0.2175+0.0660` | **0.7070** |
| step | `[0.05, 0.80, 0.55]` | `0.05×0.55=0.0275`<br>`0.80×0.87=0.6960`<br>`0.55×0.66=0.3630` | `0.0275+0.6960+0.3630` | **1.0865** |

For example, the first row is:

$$0.43(0.55) + 0.15(0.87) + 0.89(0.66) = 0.9544$$

In [ ]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55],  # step
    ]
)

query = inputs[1]
attention_scores = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attention_scores[i] = torch.dot(x_i, query)

print("Query:", query)
print("Attention scores:", attention_scores)
print("First score calculation:")
print("(0.43 * 0.55) + (0.15 * 0.87) + (0.89 * 0.66)")
print("=", attention_scores[0])

## 2. Convert scores into attention weights

Raw scores are not probabilities and do not sum to one. Softmax first applies the exponential function and then divides each value by the sum of all exponential values:

$$\text{softmax}(s_i) = \frac{e^{s_i}}{\sum_j e^{s_j}}$$

### Naive softmax calculation

The naive implementation calculates the exponentials directly:

```python
exp_scores = torch.exp(attention_scores)
naive_weights = exp_scores / exp_scores.sum()
```

The denominator is:

$$2.5971 + 4.4593 + 4.3728 + 2.3243 + 2.0279 + 2.9639 = 18.7453$$

| Token | Score $s_i$ | $e^{s_i}$ | Division by total | Naive weight |
|---|---:|---:|---:|---:|
| Your | 0.9544 | 2.5971 | `2.5971 / 18.7453` | **0.1385** |
| journey | 1.4950 | 4.4593 | `4.4593 / 18.7453` | **0.2379** |
| starts | 1.4754 | 4.3728 | `4.3728 / 18.7453` | **0.2333** |
| with | 0.8434 | 2.3243 | `2.3243 / 18.7453` | **0.1240** |
| one | 0.7070 | 2.0279 | `2.0279 / 18.7453` | **0.1082** |
| step | 1.0865 | 2.9639 | `2.9639 / 18.7453` | **0.1581** |
| **Sum** |  | **18.7453** |  | **1.0000** |

### PyTorch softmax calculation

`torch.softmax` produces the same mathematical result, but it uses a numerically stable calculation. It subtracts the largest score before applying `exp`:

```python
max_score = attention_scores.max()  # 1.4950
shifted_scores = attention_scores - max_score
stable_exp = torch.exp(shifted_scores)
attention_weights = stable_exp / stable_exp.sum()
```

Subtracting the same value from every score does not change the final softmax weights, but it prevents very large exponentials from overflowing. Here the stable denominator is:

$$0.5824 + 1.0000 + 0.9806 + 0.5212 + 0.4548 + 0.6646 = 4.2036$$

| Token | Score $s_i$ | Shifted score $s_i - 1.4950$ | $e^{s_i-1.4950}$ | Division by stable total | PyTorch weight |
|---|---:|---:|---:|---:|---:|
| Your | 0.9544 | -0.5406 | 0.5824 | `0.5824 / 4.2036` | **0.1385** |
| journey | 1.4950 | 0.0000 | 1.0000 | `1.0000 / 4.2036` | **0.2379** |
| starts | 1.4754 | -0.0196 | 0.9806 | `0.9806 / 4.2036` | **0.2333** |
| with | 0.8434 | -0.6516 | 0.5212 | `0.5212 / 4.2036` | **0.1240** |
| one | 0.7070 | -0.7880 | 0.4548 | `0.4548 / 4.2036` | **0.1082** |
| step | 1.0865 | -0.4085 | 0.6646 | `0.6646 / 4.2036` | **0.1581** |
| **Sum** |  |  | **4.2036** |  | **1.0000** |

The two implementations are mathematically equivalent:

```python
torch.allclose(naive_weights, attention_weights)
# True
```

The difference is numerical safety. The built-in `torch.softmax(attention_scores, dim=0)` should be preferred in real models.

In [ ]:
# Simple normalization, included to show the basic idea of weights.
simple_weights = attention_scores / attention_scores.sum()

# Naive softmax: exponentiate each score, then divide by the total.
exp_scores = torch.exp(attention_scores)
naive_weights = exp_scores / exp_scores.sum()

# Stable softmax calculation used conceptually by PyTorch.
max_score = attention_scores.max()
shifted_scores = attention_scores - max_score
stable_exp_scores = torch.exp(shifted_scores)
stable_weights = stable_exp_scores / stable_exp_scores.sum()

# Stable, built-in softmax. dim=0 is correct because this is a 1D tensor.
attention_weights = torch.softmax(attention_scores, dim=0)

print("Simple normalized weights:", simple_weights)
print("Naive exp values:         ", exp_scores)
print("Naive softmax weights:    ", naive_weights)
print("Max score used by stable calculation:", max_score)
print("Shifted scores:            ", shifted_scores)
print("Stable exp values:         ", stable_exp_scores)
print("Stable weights:            ", stable_weights)
print("PyTorch softmax weights:   ", attention_weights)
print("Weights sum:", attention_weights.sum())
print("Naive and PyTorch match:", torch.allclose(naive_weights, attention_weights))

## 3. Create a context vector

The context vector is a weighted sum of all input vectors:

$$z = \sum_i \alpha_i x_i$$

Each input vector is multiplied by its attention weight. Then the resulting vectors are added component by component.

| Token | Weight $\alpha_i$ | Weighted dimension 1 | Weighted dimension 2 | Weighted dimension 3 |
|---|---:|---:|---:|---:|
| Your | 0.1385 | `0.1385×0.43 = 0.0596` | `0.1385×0.15 = 0.0208` | `0.1385×0.89 = 0.1233` |
| journey | 0.2379 | `0.2379×0.55 = 0.1308` | `0.2379×0.87 = 0.2070` | `0.2379×0.66 = 0.1570` |
| starts | 0.2333 | `0.2333×0.57 = 0.1330` | `0.2333×0.85 = 0.1983` | `0.2333×0.64 = 0.1493` |
| with | 0.1240 | `0.1240×0.22 = 0.0273` | `0.1240×0.58 = 0.0719` | `0.1240×0.33 = 0.0409` |
| one | 0.1082 | `0.1082×0.77 = 0.0833` | `0.1082×0.25 = 0.0270` | `0.1082×0.10 = 0.0108` |
| step | 0.1581 | `0.1581×0.05 = 0.0079` | `0.1581×0.80 = 0.1265` | `0.1581×0.55 = 0.0870` |
| **Sum** | 1.0000 | **0.4419** | **0.6515** | **0.5683** |

The first context-vector component is the sum of the first weighted column:

$$0.0596 + 0.1308 + 0.1330 + 0.0273 + 0.0833 + 0.0079 = 0.4419$$

The other two components are calculated in exactly the same way, giving:

```text
context vector = [0.4419, 0.6515, 0.5683]
```

The loop and `attention_weights @ inputs` produce the same weighted sum.

In [ ]:
context_vector = torch.zeros_like(query)

for weight, x_i in zip(attention_weights, inputs):
    contribution = weight * x_i
    context_vector += contribution
    print(f"weight={weight:.4f}, contribution={contribution}")

print("\nContext vector from the loop:", context_vector)
print("Context vector from @:       ", attention_weights @ inputs)

## 4. Calculate self-attention for every query at once

So far, we used one query. Self-attention repeats the same process for every input row. The matrix multiplication

```python
all_attention_scores = inputs @ inputs.T
```

calculates every pairwise dot product. If `inputs` has shape `(6, 3)`, then `inputs.T` has shape `(3, 6)`, and the score matrix has shape `(6, 6)`.

### Score matrix

Each entry is:

$$S_{ij} = x_i \cdot x_j$$

For example:

$$S_{0,1} = [0.43,0.15,0.89] \cdot [0.55,0.87,0.66] = 0.9544$$

| Query \ Key | Your | journey | starts | with | one | step |
|---|---:|---:|---:|---:|---:|---:|
| Your | 0.9995 | 0.9544 | 0.9422 | 0.4753 | 0.4576 | 0.6310 |
| journey | 0.9544 | 1.4950 | 1.4754 | 0.8434 | 0.7070 | 1.0865 |
| starts | 0.9422 | 1.4754 | 1.4570 | 0.8296 | 0.7154 | 1.0605 |
| with | 0.4753 | 0.8434 | 0.8296 | 0.4937 | 0.3474 | 0.6565 |
| one | 0.4576 | 0.7070 | 0.7154 | 0.3474 | 0.6654 | 0.2935 |
| step | 0.6310 | 1.0865 | 1.0605 | 0.6565 | 0.2935 | 0.9450 |

Applying `softmax(..., dim=-1)` normalizes each row across its keys. The resulting attention-weight matrix has the same `(6, 6)` shape, and each row sums to 1. Finally, multiplying the weight matrix by `inputs` produces one context vector for each query:

```python
all_context_vectors = all_attention_weights @ inputs
```

This is the batched version of the single-query calculation from the previous section.

In [ ]:
# (6, 3) @ (3, 6) -> (6, 6): all query-key dot products.
all_attention_scores = inputs @ inputs.T

# Normalize each row across its keys. -1 means the last dimension.
all_attention_weights = torch.softmax(all_attention_scores, dim=-1)

# (6, 6) @ (6, 3) -> (6, 3): one context vector per query.
all_context_vectors = all_attention_weights @ inputs

print("All attention scores:\n", all_attention_scores)
print("\nAll attention weights:\n", all_attention_weights)
print("\nEach row sums to:", all_attention_weights.sum(dim=-1))
print("\nAll context vectors:\n", all_context_vectors)
print("\nThe second context vector matches the single-query result:")
print(torch.allclose(all_context_vectors[1], context_vector))